In [1]:
import sympy as sp

In [2]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [3]:
# define q(t)
N = 1 #number of vortexes

q = sp.Matrix()

if N==1:
    A, xc, yc = sp.symbols("A, x_c, y_c", real=True)
    L = sp.symbols("L", real=True, positive=True)
    q = sp.Matrix([A, L, xc, yc])
elif N==2:
    A1, L1, xc1, yc1 = sp.symbols("A_1, L_1, x_c_1, y_c_1", real=True)
    A2, L2, xc2, yc2 = sp.symbols("A_2, L_2, x_c_2, y_c_2", real=True)
    q = sp.Matrix([A1, L1, xc1, yc1, A2, L2, xc2, yc2])
else:
    print("error")

q

Matrix([
[  A],
[  L],
[x_c],
[y_c]])

In [4]:
# define the ansatz u_hat(x; q)
ansatz_gamma = 0
for j in range(N):
    i = j*4
    ansatz_gamma = ansatz_gamma + q[i]*sp.exp(-(((x-q[i+2])**2+(y-q[i+3])**2))/q[i+1]**2)

ansatz_gamma

A*exp((-(x - x_c)**2 - (y - y_c)**2)/L**2)

In [5]:
ansatz_u = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u

Matrix([
[-2*A*(y - y_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**2],
[ 2*A*(x - x_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**2]])

In [6]:
ansatz = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
ansatz.simplify()

4*A*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**4

In [7]:
ansatz.diff(A).simplify()

4*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp((-(x - x_c)**2 - (y - y_c)**2)/L**2)/L**4

In [8]:
ansatz.diff(L).simplify()

8*A*(L**2*(-L**2 + 2*(x - x_c)**2 + 2*(y - y_c)**2) + ((x - x_c)**2 + (y - y_c)**2)*(L**2 - (x - x_c)**2 - (y - y_c)**2))*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**7

In [9]:
sp.Derivative(ansatz, xc).simplify().doit()

4*A*(2*x - 2*x_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**4 - 4*A*(-2*x + 2*x_c)*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**6

In [10]:
# compute partial derivatives du/dqi
# du_dq = ansatz.diff(q)
du_dq = sp.Matrix([
    sp.Derivative(ansatz, A).simplify().doit(),
    sp.Derivative(ansatz, L).simplify().doit(),
    sp.Derivative(ansatz,xc).simplify().doit(),
    sp.Derivative(ansatz,yc).simplify().doit()
])

du_dq.simplify()

Matrix([
[                                                                                                                                                                              4*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**4],
[8*A*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**3 - 16*A*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**5 + 8*A*((x - x_c)**2 + (y - y_c)**2)*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**7],
[                                                                                           4*A*(2*x - 2*x_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**4 - 4*A*(-2*x + 2*x_c)*(L**2 - (x - x_c)**2 - (y - y_c)**2)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**6],
[                                                                                           4*A*(2*y - 2*y_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)/L**4 - 4*A*(-2*y + 2*y_c)*(L**2 - (x - x_c)**2 - (y - 

In [11]:
xmin,xmax = sp.symbols("x_{min}, x_{max}")
# define the inner product according to the problem
def inner_prod_H(f, g):
    return sp.integrate((f*g).expand(),(x, -sp.oo, sp.oo), (y, -sp.oo, sp.oo))

In [12]:
inner_prod_H(du_dq[3], du_dq[3]).simplify()

12*pi*A**2/L**4

In [13]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(du_dq[i], du_dq[i]).simplify()
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(du_dq[i], du_dq[j]).simplify()
        M[j,i] = M[i,j]
        print(M[i, j])



4*pi/L**2
-4*pi*A/L**3
0
0
16*pi*A**2/L**4
0
0
12*pi*A**2/L**4
0
12*pi*A**2/L**4


In [14]:
M

Matrix([
[   4*pi/L**2,    -4*pi*A/L**3,               0,               0],
[-4*pi*A/L**3, 16*pi*A**2/L**4,               0,               0],
[           0,               0, 12*pi*A**2/L**4,               0],
[           0,               0,               0, 12*pi*A**2/L**4]])

In [15]:
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [18]:
# compute rhs from the ansatz
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

0

In [19]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(du_dq[i], Fua).simplify()
    print(f[i])

f

0
0
0
0


Matrix([
[0],
[0],
[0],
[0]])

In [20]:
q_dot = M.inv()*f

q_dot.simplify()

In [21]:
q_dot

Matrix([
[0],
[0],
[0],
[0]])